# 02 - Data Preprocessing & Outlier Removal

Label fixing (one-hot to integer), IsolationForest-based outlier removal per class, clean chunk creation, and dataset distribution tables.

In [1]:
import numpy as np

# Load your label chunk (update path if necessary)
file_path = "y_val_chunk_0.npy"
labels = np.load(file_path, mmap_mode="r")

# If it's one-hot encoded, convert to 1D integer array
if len(labels.shape) > 1:
    labels = np.argmax(labels, axis=1)

# Mask out class 5, exactly as we did in training and evaluation
labels = labels[labels < 5]

# Count the occurrences of each integer label
unique_classes, counts = np.unique(labels, return_counts=True)

print(f"--- Label Distribution for {file_path} ---")
for cls, count in zip(unique_classes, counts):
    print(f"Integer Label {cls}: {count} images")
print("------------------------------------------")

--- Label Distribution for y_val_chunk_0.npy ---
Integer Label 0: 7810 images
Integer Label 1: 2412 images
Integer Label 2: 1061 images
Integer Label 3: 1495 images
Integer Label 4: 2222 images
------------------------------------------


In [3]:
import numpy as np
import glob
import os

# Put your folder names here. "." means the current folder.
folders_to_check = [".", "temp_verification_folder"]
label_files = []

# Gather all the label files from these folders
for folder in folders_to_check:
    files = glob.glob(os.path.join(folder, "y_*_chunk_*.npy"))
    label_files.extend(files)

print(f"Found {len(label_files)} label files to check...")

for file in label_files:
    lbls = np.load(file)
    
    # Check if they are the 6-column one-hot arrays we expect
    if len(lbls.shape) > 1 and lbls.shape[1] == 6:
        fixed_lbls = np.zeros(len(lbls), dtype=np.int64)
        
        for i in range(len(lbls)):
            row = lbls[i]
            # row[5] is the 'any' column. 0 means healthy brain.
            if row[5] == 0:
                fixed_lbls[i] = 0  # Class 0: Healthy
            else:
                # If there is a bleed, find it (cols 0-4) and shift by 1
                # Epidural (index 0) becomes Class 1, IPH (index 1) becomes Class 2, etc.
                fixed_lbls[i] = np.argmax(row[:5]) + 1
                
        # Overwrite the old file with the fixed integer labels
        np.save(file, fixed_lbls)
        print(f"✅ Fixed and saved: {file} (New shape: {fixed_lbls.shape})")
    else:
        print(f"⚠️ Skipping {file} - already fixed or wrong shape: {lbls.shape}")

print("\n🎉 All done! Your labels are fixed.")

Found 11 label files to check...
⚠️ Skipping ./y_val_chunk_0.npy - already fixed or wrong shape: (15000,)
✅ Fixed and saved: temp_verification_folder/y_train_chunk_0.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_9.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_1.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_2.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_10.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_11.npy (New shape: (7692,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_4.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_6.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_7.npy (New shape: (15000,))
✅ Fixed and saved: temp_verification_folder/y_train_chunk_8.npy (New shape: (15000,))

🎉 All done! Your labels are fixe

In [13]:
feat_0 = features[labels == 0]
feat_1 = features[labels == 1]

print("Class 0 samples:", feat_0.shape[0])
print("Class 1 samples:", feat_1.shape[0])


Class 0 samples: 89998
Class 1 samples: 90002


In [14]:
from sklearn.ensemble import IsolationForest

contamination = 0.05   # 5% (safe start)

iso_0 = IsolationForest(
    n_estimators=300,
    contamination=contamination,
    random_state=42,
    n_jobs=-1
)

outlier_flag_0 = iso_0.fit_predict(feat_0)   # -1 = outlier


In [15]:
num_outliers_0 = (outlier_flag_0 == -1).sum()
total_0 = feat_0.shape[0]

print(f"Class 0 outliers: {num_outliers_0} / {total_0} ({100*num_outliers_0/total_0:.2f}%)")


Class 0 outliers: 4500 / 89998 (5.00%)


In [16]:
import numpy as np

# indices of all samples
idx_all = np.arange(len(labels))

# split indices by class
idx_0 = idx_all[labels == 0]
idx_1 = idx_all[labels == 1]

# keep only NORMAL negatives
keep_0 = (outlier_flag_0 == 1)     # True = keep
clean_idx_0 = idx_0[keep_0]

# keep ALL positives
clean_idx_1 = idx_1

# final clean index list
clean_indices = np.concatenate([clean_idx_0, clean_idx_1])

print("Original samples:", len(labels))
print("After removing class-0 outliers:", len(clean_indices))


Original samples: 180000
After removing class-0 outliers: 175500


In [17]:
clean_indices.shape

(175500,)

In [21]:
import os
import numpy as np

clean_dir = "train_data_clean"
os.makedirs(clean_dir, exist_ok=True)

new_chunk_id = 0
class0_ptr = 0   # pointer into outlier_flag_0 (VERY IMPORTANT)

for imgs, lbls in train_loader:   # batch_size=1 → one chunk
    imgs = np.array(imgs)         # (N,H,W,C)
    lbls = np.array(lbls).reshape(-1)

    keep_mask = np.ones(len(lbls), dtype=bool)

    for i, y in enumerate(lbls):
        if y == 0:
            # check outlier flag for THIS class-0 sample
            if outlier_flag_0[class0_ptr] == -1:
                keep_mask[i] = False   # REMOVE outlier
            class0_ptr += 1   # advance pointer ONLY for class-0

        # if y == 1 → always keep

    clean_imgs = imgs[keep_mask]
    clean_lbls = lbls[keep_mask]

    if len(clean_lbls) > 0:
        np.save(f"{clean_dir}/chunk_{new_chunk_id:03d}.npy", clean_imgs)
        np.save(f"{clean_dir}/labels_chunk_{new_chunk_id:03d}.npy", clean_lbls)
        new_chunk_id += 1

print("✅ Clean chunks created:", new_chunk_id)
print("Class-0 pointer ended at:", class0_ptr)
print("Expected class-0 samples:", len(outlier_flag_0))


✅ Clean chunks created: 12
Class-0 pointer ended at: 89998
Expected class-0 samples: 89998


In [22]:
clean_train_dataset = NumpyDataset("train_data_clean")

total = 0
count_0 = 0
count_1 = 0

for imgs, lbls in clean_train_dataset:
    lbls = np.array(lbls)
    total += len(lbls)
    count_0 += (lbls == 0).sum()
    count_1 += (lbls == 1).sum()

print("Total samples:", total)
print("Class 0:", count_0)
print("Class 1:", count_1)


Total samples: 175500
Class 0: 85498
Class 1: 90002


In [11]:
import os
import numpy as np
import pandas as pd

def count_labels(directory, prefix):
    counts = np.zeros(6, dtype=int)
    
    # Safely check if directory exists
    if not os.path.exists(directory):
        print(f"⚠️ Warning: Folder '{directory}' not found. Returning zeros.")
        return counts
        
    files = sorted([f for f in os.listdir(directory) if f.startswith(prefix) and f.endswith(".npy")])
    
    if len(files) == 0:
        print(f"⚠️ Warning: No files starting with '{prefix}' found in '{directory}'.")
        return counts

    for f in files:
        labels = np.load(os.path.join(directory, f), mmap_mode="r")
        
        # Handle one-hot/multi-label (2D array) vs sparse/argmax (1D array)
        if labels.ndim > 1 and labels.shape[-1] == 6:
            counts += np.sum(labels == 1, axis=0).astype(int)
        else:
            unique, iter_counts = np.unique(labels, return_counts=True)
            for val, count in zip(unique, iter_counts):
                counts[int(val)] += count
                
    return counts

# The 6 classes for your RSNA XAI-ICH project
class_names = ["Healthy / Any", "Epidural (EPH)", "Intraparenchymal (IPH)", 
               "Intraventricular (IVH)", "Subarachnoid (SAH)", "Subdural (SDH)"]

print("Scanning data folders... This might take a minute depending on your hard drive speed.")

# === FOLDER PATHS ===
# 🛑 CHANGE 'raw_training_data' to whatever your pre-cleaned folder is named!
raw_train_folder = "temp_verification_folder"      
clean_train_folder = "clean_training_data"  
val_folder = "." # Current directory for validation files

# Get the counts for all three columns
raw_counts = count_labels(raw_train_folder, "y_train_chunk_")
clean_counts = count_labels(clean_train_folder, "y_train_chunk_")
val_counts = count_labels(val_folder, "y_val_chunk_")

# Build the DataFrame
df = pd.DataFrame({
    "ICH Subtypes": class_names,
    "Train (Before Outliers)": raw_counts,
    "Train (After Outliers)": clean_counts,
    "Validation / Test": val_counts
})

# Calculate the totals for the bottom row
df.loc[6] = ["Total", raw_counts.sum(), clean_counts.sum(), val_counts.sum()]

# Print the final publication-ready table
print("\n" + "="*85)
print("TABLE 1: DATASET DISTRIBUTION BEFORE AND AFTER OUTLIER REMOVAL")
print("="*85)
print(df.to_string(index=False))
print("="*85)

Scanning data folders... This might take a minute depending on your hard drive speed.

TABLE 1: DATASET DISTRIBUTION BEFORE AND AFTER OUTLIER REMOVAL
          ICH Subtypes  Train (Before Outliers)  Train (After Outliers)  Validation / Test
         Healthy / Any                    71271                   67707               7616
        Epidural (EPH)                     2116                    2116                194
Intraparenchymal (IPH)                    23458                   22285               2412
Intraventricular (IVH)                    10319                    9803               1061
    Subarachnoid (SAH)                    14184                   13474               1495
        Subdural (SDH)                    21344                   20276               2222
                 Total                   142692                  135661              15000


In [7]:
import os
import numpy as np
import pandas as pd

def count_folder(directory, prefix="labels_chunk"):
    counts = {"Healthy": 0, "Hemorrhage": 0}
    if not os.path.exists(directory): return counts
        
    files = [f for f in os.listdir(directory) if f.startswith(prefix) and f.endswith(".npy")]
    for f in files:
        labels = np.load(os.path.join(directory, f), mmap_mode="r")
        if labels.ndim > 1 and labels.shape[-1] == 6:
            counts["Healthy"] += np.sum(labels[:, 0] == 1)
            counts["Hemorrhage"] += np.sum(np.max(labels[:, 1:], axis=1) == 1)
        else:
            counts["Healthy"] += np.sum(labels == 0)
            counts["Hemorrhage"] += np.sum(labels > 0)
    return counts

def count_file(file_path):
    counts = {"Healthy": 0, "Hemorrhage": 0}
    if not os.path.exists(file_path): return counts
        
    labels = np.load(file_path, mmap_mode="r")
    if labels.ndim > 1 and labels.shape[-1] == 6:
        counts["Healthy"] += np.sum(labels[:, 0] == 1)
        counts["Hemorrhage"] += np.sum(np.max(labels[:, 1:], axis=1) == 1)
    else:
        counts["Healthy"] += np.sum(labels == 0)
        counts["Hemorrhage"] += np.sum(labels > 0)
    return counts

# ==========================================
# PATHS
raw_train_folder = "train_data"            
clean_train_folder = "train_data_clean"    
val_label_file = "val_label.npy"               
# ==========================================

print("Extracting Binary Dataset Distribution...")

# Using the correct prefix from your screenshot!
raw_counts = count_folder(raw_train_folder, prefix="labels_chunk")
clean_counts = count_folder(clean_train_folder, prefix="labels_chunk")
val_counts = count_file(val_label_file)

df = pd.DataFrame({
    "Binary Class": ["Healthy (Negative)", "Hemorrhage (Positive)"],
    "Train (Before Outliers)": [raw_counts["Healthy"], raw_counts["Hemorrhage"]],
    "Train (After Outliers)": [clean_counts["Healthy"], clean_counts["Hemorrhage"]],
    "Validation / Test": [val_counts["Healthy"], val_counts["Hemorrhage"]]
})

df.loc[2] = ["Total Slices", sum(raw_counts.values()), sum(clean_counts.values()), sum(val_counts.values())]

print("\n" + "="*85)
print("TABLE 2: BINARY DATASET DISTRIBUTION")
print("="*85)
print(df.to_string(index=False))
print("="*85)

Extracting Binary Dataset Distribution...

TABLE 2: BINARY DATASET DISTRIBUTION
         Binary Class  Train (Before Outliers)  Train (After Outliers)  Validation / Test
   Healthy (Negative)                    89998                   85498               7544
Hemorrhage (Positive)                    90002                   90002               7456
         Total Slices                   180000                  175500              15000
